In [ ]:
%matplotlib inline
import os
os.environ['PY3_PROD'] = '1'
%load_ext autoreload
%autoreload 2
os.system('kinit')

In [ ]:
import numpy as np
import pandas as pd
import datetime
import matplotlib
from pycmqlib3.utility import dbaccess, dataseries, misc
from pycmqlib3.analytics.tstool import *
from pycmqlib3.analytics.btmetrics import *
from pycmqlib3.analytics.backtest_utils import *
from pycmqlib3.strategy.signal_repo import *

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
matplotlib.rcParams['figure.figsize'] = (12, 8)
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>div.output_scroll { height: 44em; }</style>"))

# define product groups and start times

In [ ]:
ferrous_products_mkts = ['rb', 'hc', 'i', 'j', 'jm']
ferrous_mixed_mkts = ['ru', 'FG', 'SM', "SF", 'nr', 'SA', 'UR'] # 'ZC', 
base_metal_mkts = ['cu', 'al', 'zn', 'pb', 'ni', 'sn', 'ss', 'ao', 'si']
precious_metal_mkts = ['au', 'ag']
ind_metal_mkts = ferrous_products_mkts + ferrous_mixed_mkts + base_metal_mkts  
petro_chem_mkts = ['l', 'pp', 'v', 'TA', 'MA', 'bu', 'sc', 'fu', 'eg', 'eb', 'lu', 'pg', 'PF'] 
ind_all_mkts = ind_metal_mkts + petro_chem_mkts
ags_oil_mkts = ['m', 'RM', 'y', 'p', 'OI', 'a', 'c', 'cs', 'b'] #, 'b']
ags_soft_mkts = ['CF', 'SR', 'jd', 'AP', 'sp', 'CJ', 'lh', 'PK', 'CY'] # 'CY',] 

ags_all_mkts = ags_oil_mkts + ags_soft_mkts

eq_fut_mkts = ['IF', 'IH', 'IC']
bond_fut_mkts = ['T', 'TF', 'TS']

fin_all_mkts = eq_fut_mkts + bond_fut_mkts
commod_all_mkts = ind_all_mkts + ags_all_mkts + precious_metal_mkts
all_markets = commod_all_mkts + fin_all_mkts

# load historical data

In [ ]:
from misc_scripts.update_fut_prices import load_saved_fut

tday = datetime.date(2025,7, 11)
df = load_saved_fut(tday, freq='d')
#df = load_cnc_fut(tday, type='cal')

spot_df = load_fun_data(tday)

start_date = df.index[0]
end_date = tday

cdates = pd.date_range(start=start_date, end=tday, freq='D')
bdates = pd.bdate_range(start=start_date, end=end_date, freq='C', holidays=misc.CHN_Holidays)

In [ ]:
# bdates = [d for d in bdates if d.date() not in [datetime.date(2014,1,2), datetime.date(2014,1,3)]]
# df = df.reindex(index=bdates)

In [ ]:
#product_list = df.
vol_win = 20
product_list = list(set([col[:-2] for col in df.columns.get_level_values(0).unique()]))

spot_dict = {}

for asset in product_list:    
    if (asset+'c2', 'close') not in df.columns:
        print(asset)
        continue
    spot_dict[f'{asset}_ryield'] = (np.log(df[(asset+'c1', 'close')]) - 
                                  np.log(df[(asset+'c2', 'close')]) - 
                                  df[(asset+'c1', 'shift')] + 
                                  df[(asset+'c2', 'shift')])/(df[(asset+'c2', 'expiry')] - 
                                                              df[(asset+'c1', 'expiry')]).dt.days*365.0 + spot_df['r007_cn'].dropna()/100
    spot_dict[f'{asset}_px'] = df[(asset+'c1', 'close')]
    spot_dict[f'{asset}_px_unadj'] = df[(asset+'c1', 'close')]/np.exp(df[(asset+'c1', 'shift')])
    spot_dict[f'{asset}_logret'] = np.log(df[(asset+'c1', 'close')]).dropna().diff()
    spot_dict[f'{asset}_colr'] = np.log(df[(asset+'c1', 'close')]/df[(asset+'c1', 'open')])
    spot_dict[f'{asset}_pctchg'] = df[(asset+'c1', 'close')].dropna().pct_change()    
    spot_dict[f'{asset}_logret2'] = np.log(df[(asset+'c2', 'close')]).dropna().diff()
    spot_dict[f'{asset}_basmom'] = spot_dict[f'{asset}_logret'] - spot_dict[f'{asset}_logret2']        
    spot_dict[f'{asset}_pctvol'] = spot_dict[f'{asset}_pctchg'].dropna().rolling(vol_win).std()
    spot_dict[f'{asset}_basmom5'] = spot_dict[f'{asset}_basmom'].dropna().rolling(5).sum()
    spot_dict[f'{asset}_basmom10'] = spot_dict[f'{asset}_basmom'].dropna().rolling(10).sum()
    spot_dict[f'{asset}_basmom20'] = spot_dict[f'{asset}_basmom'].dropna().rolling(20).sum()
    spot_dict[f'{asset}_basmom40'] = spot_dict[f'{asset}_basmom'].dropna().rolling(40).sum()
    spot_dict[f'{asset}_basmom60'] = spot_dict[f'{asset}_basmom'].dropna().rolling(60).sum()
    spot_dict[f'{asset}_basmom100'] = spot_dict[f'{asset}_basmom'].dropna().rolling(100).sum()
    spot_dict[f'{asset}_basmom120'] = spot_dict[f'{asset}_basmom'].dropna().rolling(120).sum()
    spot_dict[f'{asset}_basmom170'] = spot_dict[f'{asset}_basmom'].dropna().rolling(170).sum()
    spot_dict[f'{asset}_basmom180'] = spot_dict[f'{asset}_basmom'].dropna().rolling(180).sum()
    
spot_df = pd.concat([spot_df, pd.DataFrame(spot_dict)], axis=1)

# feature study

In [ ]:
vol_win=20
pnl_tenors = ['6m', '1y', '2y', '3y', '4y', '5y', '6y', '7y', '8y', '9y', '10y', '12y', '15y']

insample = '2026-01-01'

empiric_assets = [
    'rb', 'hc', 'i', 'j', 'jm', 
    'FG', 'v', 'SM', 'SF', 'SA', 'UR',
    'cu', 'al', 'zn', 'ni', 'pb', 'sn', 'ss', 'ao', 
    'au', 'ag', #'bc', 
    #'si', 'lc',
    'ru', 'l', 'pp', 'TA', 'MA', 'sc', 'eb', 'eg', 'lu', 'bu', 'fu',
    'm', 'RM', 'y', 'p', 'OI', 'a', 'c', 'CF', 'jd', 'AP', 'lh', #'SR', #'cs', #'PK', 'CJ', 'sp',
]

traded_price = 'n305'
df_pxchg = pd.DataFrame(index=df.index)[:insample]
for asset in empiric_assets:
    ts_px = df[(asset+'c1', traded_price)]
    if traded_price in ['n305', 'n310', 'n315', 'n450']:
        flag = ts_px.isna()
        ts_px.loc[flag] = df[(asset+'c1', 'a1505')].loc[flag]
    df_pxchg[asset] = ts_px.dropna().pct_change()[:insample]
vol_df = df_pxchg.rolling(vol_win).std()

customerized signal

In [ ]:
cutoff='2010-01-01'

shift_holdings = 1
signal_cap = [-2.0, 2.0]
chg_func = 'diff'
bullish = True
vol_win = 20
signal_func = 'sgn_ma'
param_rng = [20, 40, 1]
param_args = {} #{"h1s": [1, 5, 1]}
feature = 'logret'

signal_df = pd.DataFrame(index=df_pxchg.index)
signal_dict = {}
for asset in empiric_assets:
    feature_ts = spot_df[f'{asset}_{feature}'].dropna()
    #feature_ts = df[(asset+'c1', 'close')].dropna().pct_change()
    # tmp_df = df.loc[:, df.columns.get_level_values(0) == asset+'c1']
    # vol_ts = gk_vol_est(tmp_df, window = vol_win, trading_periods=252, clean=True)
    #feature_ts = np.log(df[(asset+'c1', 'close')]/df[(asset+'c1', 'close')].shift(1))
    #feature_ts = (2*(df[(asset+'c1', 'high')] - df[(asset+'c1', 'low')]) * np.sign(df[(asset+'c1', 'close')] - df[(asset+'c1', 'open')]) - (df[(asset+'c1', 'close')] - df[(asset+'c1', 'open')]))/df[(asset+'c1', 'close')]
    #signal_ts = feature_ts.rolling(10).mean()
    #ticker, param_rng = feature_map[asset]    
    
    #signal_ts = conv_ewm(feature_ts, param_args['h1s'], param_rng, vol_win)
    #feature_ts = spot_df[f'{asset}_px_unadj'].dropna().rolling(60).mean().shift(900)/spot_df[f'{asset}_px_unadj'].dropna() - 1
    #feature_ts = feature_ts.dropna()    
    # if len(feature_ts) ==0:
    #     signal_df[asset] = np.nan
    #     continue
#     feature_ts = feature_ts.pct_change()
#     feature_ts = feature_ts/feature_ts.rolling(60).std()
#     feature_ts = feature_ts.cumsum()
    #feature_ts = spot_df[feature].ffill().reindex(index=pd.date_range(start=df.index[0], end=df.index[-1], freq=freq)).ffill().dropna()
    #feature_ts = spot_df[f'{asset}_{feature}'].dropna()        
    #feature_ts = np.log(feature_ts)
#     if asset in ['cu', 'al']:
#         #param_rng = [1, 2, 1]
#         feature_ts = feature_ts.rolling(20).mean()
#     else:
#         #param_rng = [1, 2, 1]
#         feature_ts = feature_ts.rolling(2).mean()
    #feature_ts = feature_ts.reindex(index=cdates).ffill().reindex(index=bdates)
    #feature_ts = feature_ts.ewm(1).mean()
    #feature_ts = yoy_generic(feature_ts, label_func=lunar_label, group_col='label_day', func=chg_func)
    #signal_ts = feature_ts
    signal_ts = calc_conv_signal(feature_ts, signal_func=signal_func, param_rng=param_rng, signal_cap=signal_cap, vol_win=vol_win, param_args=param_args)
    #signal_ts = hlratio(feature_ts, 250)
    #signal_ts = signal_hysteresis(signal_ts, 0.75, 0.25)
    signal_ts = signal_ts.ewm(3).mean()
    # signal_ts = seasonal_score(feature_ts.to_frame(), backward=10, forward=10, rolling_years=3, min_obs=10)
    # signal_ts = seasonal_score(feature_ts.to_frame(), backward=15, forward=15, rolling_years=3, min_obs=30)
    if not bullish:
        signal_ts = -signal_ts    
    #signal_ts = signal_ts.reindex(index=cdates).ffill().reindex(index=df_pxchg.index).ffill()        
    #signal_ts = signal_hump(signal_ts, 0.2)
    
    if '-' in asset:
        sub_assets = asset.split('-')
        if sub_assets[0] in signal_df.columns:
            signal_df[sub_assets[0]] += signal_ts
        else:
            signal_df[sub_assets[0]] = signal_ts
        if sub_asset[1] in signal_df.columns:
            signal_df[sub_assets[1]] -= signal_ts
        else:
            signal_df[sub_assets[1]] = -signal_ts            
    else:
        signal_df[asset] = signal_ts

signal_df = signal_df*0.5 + xs_demean(signal_df)*0.5
#signal_df = xs_rank(signal_df, 0.1)
#signal_df = xs_demean(signal_df)
#signal_df = signal_buffer(signal_df, 0.3)
# signal_df = signal_cost_optim(signal_df, 0.4, vol_df, 
#                            cost_dict = dict([(asset, 3e-4) for asset in empiric_assets]), power=3)

holding = generate_holding_from_signal(signal_df, vol_df, risk_scaling=1.0, asset_scaling=False)
bt_metrics = MetricsBase(holdings=holding[empiric_assets][cutoff:],
                         returns=df_pxchg[empiric_assets][cutoff:], 
                         shift_holdings=shift_holdings)
bt_metrics_w_cost = MetricsBase(holdings=holding[empiric_assets][cutoff:],
                                returns=df_pxchg[empiric_assets][cutoff:], 
                                shift_holdings=shift_holdings,
                                cost_dict=simple_cost(holding.columns, trd_cost=2e-4))

pnl_stats = bt_metrics.calculate_pnl_stats(shift=0, use_log_returns=False, tenors=pnl_tenors, perf_metrics=['sharpe', 'std', 'sortino', 'calmar', 'skew', 'uptail', 'lowtail'])
pnl_stats_w_cost = bt_metrics_w_cost.calculate_pnl_stats(shift=0, use_log_returns=False, tenors=pnl_tenors, perf_metrics=['sharpe', 'std', 'sortino', 'calmar', 'skew', 'uptail', 'lowtail'])
perf_stats = transform_output(pnl_stats_w_cost, metrics=['sharpe', 'std', 'sortino', 'calmar', 'skew', 'uptail', 'lowtail'])
print(perf_stats.round(2))
print("SR after cost:\n", pnl_stats_w_cost['sharpe'])
print(pnl_stats_w_cost['asset_sharpe_stats'])
print("Turnover: \n%s\nPNL per trade:\n%s\n" % (pnl_stats_w_cost['turnover'], pnl_stats_w_cost['pnl_per_trade']))

print("SR before cost:\n", pnl_stats['sharpe'])
print(pnl_stats['asset_sharpe_stats'])
print("Turnover: \n%s\nPNL per trade:\n%s\n" % (pnl_stats['turnover'], pnl_stats['pnl_per_trade']))

iplot(pnl_stats_w_cost['portfolio_cumpnl'], title='portfolio pnl with cost')
iplot(pnl_stats_w_cost['asset_cumpnl'], title='asset pnl with cost')

iplot(pnl_stats['portfolio_cumpnl'], title='portfolio pnl wo cost')
iplot(pnl_stats['asset_cumpnl'], title='asset pnl wo cost')


In [ ]:
broad_mkts = [
    'rb', 'hc', 'i', 'j', 'jm', 'FG', 'v', 'SM', 'SF', 'SA',
    'cu', 'al', 'zn', 'ni', 'pb', 'sn', 'ss', 'si', 'ao', 'au', 'ag',#'bc'
    'ru', 'l', 'pp', 'TA', 'MA', 'sc', 'eb', 'eg', 'UR', 'lu', #'bu', 'fu',
    'm', 'RM', 'y', 'p', 'OI', 'a', 'c', 'CF', 'jd', 'AP', 'lh', #'cs',
]

ind_mkts = ['rb', 'hc', 'i', 'j', 'jm', 'FG', 'v', 'SA', 'UR', 'SM', 'SF', 
    'cu', 'al', 'zn', 'ni', 'pb', 'sn', 'ss', 'ao', 'au', 'ag',#'bc', 'si', 
    'ru', 'l', 'pp', 'TA', 'MA', 'sc', 'eb', 'eg', 'lu', #'bu', 'fu',
]

ag_mkts = [
    'm', 'RM', 'y', 'p', 'OI', 'a', 'c', 'CF', 'jd', 'AP', 
]

feature_setup = {
    "ryield_ema": [broad_mkts, ["ryield", "ema", [1, 2, 1], "ema1", "", True, "", "", 60, [-2, 2]]],
    "ryield_ema_xdemean": [ind_mkts, ["ryield", "ema", [1, 2, 1], "ema1", "", True, "", "", 60, [-2, 2]]], 
    #"ryield_ema_ags_ts": [ag_mkts, ["ryield", "ema", [1, 2, 1], "ema1", "", True, "", "", 60, [-2, 2]]], # SR 0.75
    
    "ryield_st_zsa": [broad_mkts, ["ryield", "zscore_adj", [20, 30, 1], "ema1", "", True, "", "", 240, [-2, 2]]],
    "ryield_st_zsa_xdemean": [broad_mkts, ["ryield", "zscore_adj", [20, 30, 1], "ema1", "", True, "", "", 240, [-2, 2]]],
    "ryield_lt_zsa": [broad_mkts, ["ryield", "zscore_adj", [80, 120, 2], "ema1", "", True, "", "", 240, [-2, 2]]],
    "ryield_lt_zsa_xdemean": [broad_mkts, ["ryield", "zscore_adj", [80, 120, 2], "ema1", "", True, "", "", 240, [-2, 2]]],
    
    "basmom5_ema": [broad_mkts, ["basmom5", "ema", [10, 20, 1], "", "", True, "price", "", 240, [-2, 2]]],
    "basmom5_ema_xdemean": [broad_mkts, ["basmom5", "ema", [10, 20, 1], "", "", True, "price", "", 240, [-2, 2]]],
    "basmom10_ema": [broad_mkts, ["basmom10", "ema", [10, 20, 1], "", "", True, "price", "", 240, [-2, 2]]],
    "basmom10_ema_xdemean": [broad_mkts, ["basmom10", "ema", [10, 20, 1], "", "", True, "price", "", 240, [-2, 2]]],
    "basmom10_qtl": [broad_mkts, ["basmom10", "qtl", [120, 140, 2], "ema5", "", True, "price", "", 240, [-2, 2]]],
    "basmom10_qtl_xdemean": [broad_mkts, ["basmom10", "qtl", [120, 140, 2], "ema5", "", True, "price", "", 240, [-2, 2]]],
    "basmom20_ema": [broad_mkts, ["basmom20", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom20_ema_xdemean": [broad_mkts, ["basmom20", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom20_qtl": [broad_mkts, ["basmom20", "qtl", [230, 250, 2], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom20_qtl_xdemean": [broad_mkts, ["basmom20", "qtl", [230, 250, 2], "", "", True, "price", "ema1", 240, [-2, 2]]],    
    "basmom40_ema": [broad_mkts, ["basmom40", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom40_ema_xdemean": [broad_mkts, ["basmom40", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],    
    "basmom60_ema": [broad_mkts, ["basmom60", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom60_ema_xdemean": [broad_mkts, ["basmom60", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom120_ema": [broad_mkts, ["basmom120", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom120_ema_xdemean": [broad_mkts, ["basmom120", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom180_ema": [broad_mkts, ["basmom180", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],
    "basmom180_ema_xdemean": [broad_mkts, ["basmom180", "ema", [1, 2, 1], "", "", True, "price", "ema1", 240, [-2, 2]]],    

    "mom_ewmac": [broad_mkts, ["px", "ewmac", [1, 4, 1], "", "", True, "price", "ema1", 40, [-2, 2]]],
    "mom_ewmac_xdemean": [broad_mkts, ["px", "ewmac", [1, 4, 1], "", "", True, "price", "ema1", 40, [-2, 2]]],
    "mom_momma240": [broad_mkts, ["px", "ema", [1, 2, 1], "df240", "pct_change", True, "price", "ema1", 60, [-2, 2]]],
    "mom_momma240_xdemean": [broad_mkts, ["px", "ema", [1, 2, 1], "df240", "pct_change", True, "price", "ema1", 60, [-2, 2]]],
    "mom_momma20": [broad_mkts, ["px", "ema", [1, 2, 1], "df20", "pct_change", True, "price", "ema1", 60, [-2, 2]]],
    "mom_momma20_xdemean": [broad_mkts, ["px", "ema", [1, 2, 1], "df20", "pct_change", True, "price", "ema1", 60, [-2, 2]]], 

    "mom_hlr_st": [broad_mkts, ["px", "hlratio", [10, 20, 2], "", "", True, "", "buf0.3", 120, [-2, 2]]],
    "mom_hlr_st_xdemean": [broad_mkts, ["px", "hlratio", [10, 20, 2], "", "", True, "", "buf0.3", 120, [-2, 2]]],
    "mom_hlr_mt": [broad_mkts, ["px", "hlratio", [40, 60, 2], "", "", True, "", "buf0.1", 120, [-2, 2]]],
    "mom_hlr_mt_xdemean": [broad_mkts, ["px", "hlratio", [40, 60, 2], "", "", True, "", "buf0.1", 120, [-2, 2]]],
    "mom_hlr_lt": [broad_mkts, ["px", "hlratio", [80, 120, 2], "", "", True, "", "buf0.1", 120, [-2, 2]]],
    "mom_hlr_lt_xdemean": [broad_mkts, ["px", "hlratio", [80, 120, 2], "", "", True, "", "buf0.1", 120, [-2, 2]]],  
    "mom_hlr_yr": [broad_mkts, ["px", "hlratio", [240, 250, 2], "", "", True, "", "", 120, [-2, 2]]],
    "mom_hlr_yr_xdemean": [broad_mkts, ["px", "hlratio", [240, 250, 2], "", "", True, "", "", 120, [-2, 2]]],  
    "cclr_mom_sgnma": [broad_mkts, ["logret", "sgn_ma", [20, 40, 2], "", "", True, "price", "ema3", 20, [-2, 2]]],
    "cclr_mom_sgnma_xdemean": [broad_mkts, ["logret", "sgn_ma", [20, 40, 2], "", "", True, "price", "ema3", 20, [-2, 2]]],
    "colr_mom_sgnma": [broad_mkts, ["colr", "sgn_ma", [20, 40, 2], "", "", True, "price", "ema3", 20, [-2, 2]]],
    "colr_mom_sgnma_xdemean": [broad_mkts, ["colr", "sgn_ma", [20, 40, 2], "", "", True, "price", "ema3", 20, [-2, 2]]],
}
    

standard signal

In [ ]:
cutoff='2010-01-01'

risk_scaling = 1
feature_names = [
    ["ryield_ema", 0.082], #1.5
    ["ryield_ema_xdemean", 0.20], # 4.8
    #["ryield_ema_ags_ts", 0.5],
    
    ["ryield_st_zsa", 0.016], # 0.6
    ["ryield_st_zsa_xdemean", 0.02], # 2.4
    ["ryield_lt_zsa", 0.025], # 0.5
    ["ryield_lt_zsa_xdemean", 0.045], # 1.4
    
#     ["basmom5_ema", 1.0],
#     ["basmom5_ema_xdemean", 1.0],
#     ["basmom10_ema", 1.0],
#     ["basmom10_ema_xdemean", 1.0],    
    ["basmom20_ema", 0.013],
    ["basmom20_ema_xdemean", 0.032],           
    #["basmom40_ema", 0],
    #["basmom40_ema_xdemean", 0.0], 
    ["basmom60_ema", 0.022],
    ["basmom60_ema_xdemean", 0.027], 
    ["basmom120_ema", 0.02],
    ["basmom120_ema_xdemean", 0.035],

    #["mom_ewmac", 1.0],
    #["mom_ewmac_xdemean", 1.0],
    ["mom_momma240", 0.02],
    ["mom_momma240_xdemean", 0.062],
    ["mom_momma20", 0.009],
    ["mom_momma20_xdemean", 0.0105], 
    
    ["mom_hlr_st", 0.062],
    ["mom_hlr_st_xdemean", 0.04], 
    #["mom_hlr_mt", 0.0],
    #["mom_hlr_mt_xdemean", 1.0],    
    #["mom_hlr_lt", 0.0],
    ["mom_hlr_lt_xdemean", 0.014],    
    #["mom_hlr_yr", 0],
    #["mom_hlr_yr_xdemean", 0.0065],    

    ["cclr_mom_sgnma", 1],
    ["cclr_mom_sgnma_xdemean", 1],
    ["colr_mom_sgnma", 1],
    ["colr_mom_sgnma_xdemean", 1],    
]

signal_dict = {}
signal_df = pd.DataFrame(0, columns=empiric_assets, index=cdates)

for feature_name, weight in feature_names:
    signal_assets = feature_setup[feature_name][0]
    feature, signal_func, param_rng, proc_func, chg_func, bullish, freq, post_func, vol_win, signal_cap = feature_setup[feature_name][1] 
    if feature in ['sinv', 'base_phybas', 'prem_bonded_warrant', 'prem_bonded_cif', 
                   'tc', 'phycarry', 'ryield', 'basmom5', 'basmom10', 'basmom20', 'basmom40', 'basmom60', 'basmom120', 
                   'base_inv', 'inv_shfe_d', 'inv_lme_total', 'inv_exch_d', 'px', 'logret', 'colr']:
        sig_df = pd.DataFrame(0, columns=empiric_assets, index=cdates)
        for asset in empiric_assets:
            if feature == 'base_inv':
                asset_feature = base_inv.get(asset, f"{asset}_{feature}")
            else:
                asset_feature = f"{asset}_{feature}"
            if (asset not in signal_assets) or (asset_feature not in spot_df.columns):
                sig_df[asset] = np.nan
                continue
            if feature in ['base_phybas']:
                if asset in ['cu', 'al']: 
                    proc_func = 'sma20'
                else:
                    proc_func = 'sma2'
            signal_ts = calc_funda_signal(spot_df, asset_feature, signal_func, param_rng,
                                          proc_func=proc_func, chg_func=chg_func, bullish=bullish,
                                          freq=freq, signal_cap=signal_cap, bdates=bdates,
                                          post_func=post_func, vol_win=vol_win)
            sig_df[asset] = signal_ts
        sig_df = sig_df.reindex(index=cdates).ffill().reindex(index=df_pxchg.index)
        if "xdemean" in feature_name:
            sig_df = xs_demean(sig_df)
        elif "xscore" in feature_name:
            sig_df = xs_score(sig_df)
        elif "xrank" in feature_name:
            sig_df = xs_rank(sig_df, 0.2)
        last_func = post_func.split('|')[-1]
        if 'buf' in last_func:
            buffer_size = float(last_func[3:])
            sig_df = signal_buffer(sig_df, buffer_size)
        elif 'bfc' in last_func:
            buffer_size = float(last_func[3:])
            sig_df = signal_cost_optim(sig_df, 
                                       buffer_size, 
                                       vol_df, 
                                       cost_dict = dict([(asset, 3e-4) for asset in empiric_assets]))
    else:
        sig_ts = calc_funda_signal(spot_df, feature, signal_func, param_rng,
                                   proc_func=proc_func, chg_func=chg_func, bullish=bullish,
                                   freq=freq, signal_cap=signal_cap, bdates=bdates,
                                   post_func=post_func, vol_win=vol_win)
        sig_df = pd.DataFrame(columns=empiric_assets, index=cdates)
        for asset in empiric_assets:
            if asset in signal_assets:
                sig_df[asset] = sig_ts
            else:
                sig_df[asset] = 0
    sig_df = sig_df.reindex(index=cdates).ffill().reindex(index=df_pxchg.index).fillna(0)
    signal_dict[feature_name] = sig_df * weight
    signal_df = signal_df + signal_dict[feature_name]
    
signal_dict['combo'] = signal_df.dropna().ffill()

In [ ]:
pnl_df = pd.DataFrame(index=df_pxchg.index)
for signal_name in signal_dict:
    signal_df = signal_dict[signal_name]
    holding = generate_holding_from_signal(signal_df, vol_df, risk_scaling=risk_scaling, asset_scaling=True)    
    
    bt_metrics = MetricsBase(holdings=holding[empiric_assets][cutoff:],
                             returns=df_pxchg[empiric_assets][cutoff:], 
                             shift_holdings=1)

    bt_metrics_w_cost = MetricsBase(holdings=holding[empiric_assets][cutoff:],
                                    returns=df_pxchg[empiric_assets][cutoff:], 
                                    shift_holdings=1,
                                    cost_dict=simple_cost(holding.columns, trd_cost=2e-4))

    pnl_stats = bt_metrics.calculate_pnl_stats(shift=0, use_log_returns=False, tenors=pnl_tenors, perf_metrics=['sharpe', 'std', 'sortino', 'calmar'])
    pnl_stats_w_cost = bt_metrics_w_cost.calculate_pnl_stats(shift=0, use_log_returns=False, tenors=pnl_tenors, perf_metrics=['sharpe', 'std', 'sortino', 'calmar'])
    pnl_df[signal_name] = pnl_stats_w_cost['portfolio_pnl']['total']
    
    print("\nsignal=%s\n" % signal_name)
    perf_stats = transform_output(pnl_stats_w_cost)
    print(perf_stats.round(2))
    print("SR after cost:\n", pnl_stats_w_cost['sharpe'])
    print(pnl_stats_w_cost['asset_sharpe_stats'])
    print("Turnover: \n%s\nPNL per trade:\n%s\n" % (pnl_stats_w_cost['turnover'], pnl_stats_w_cost['pnl_per_trade']))
    


    print("SR before cost:\n", pnl_stats['sharpe'])
    print(pnl_stats['asset_sharpe_stats'])
    print('pnl per trade - turnover: \n%s\n' % (pd.concat([pnl_stats['pnl_per_trade'].to_frame("bias"), pnl_stats['turnover'].to_frame("TO")], axis=1)))

    iplot(pnl_stats_w_cost['portfolio_cumpnl'], title='portfolio pnl with cost')
    iplot(pnl_stats_w_cost['asset_cumpnl'], title='asset pnl with cost')

    iplot(pnl_stats['portfolio_cumpnl'], title='portfolio pnl wo cost')
    iplot(pnl_stats['asset_cumpnl'], title='asset pnl wo cost')

corr_cutoff = '2017-01-01'
print("std:\n%s\n" % pnl_df[corr_cutoff:].std(axis=0))

pnl_w_df = pnl_df.drop(columns=['combo']).resample('W').sum()
print("corr:\n%s\n" % pnl_w_df[corr_cutoff:].corr())

In [ ]:
pnl_df['carry'] = pnl_df[['ryield_ema', 'ryield_ema_xdemean']].sum(axis=1)
pnl_df['carrymom'] = pnl_df[["ryield_st_zsa", "ryield_st_zsa_xdemean", "ryield_lt_zsa", "ryield_lt_zsa_xdemean", "basmom20_ema", "basmom20_ema_xdemean"]].sum(axis=1)
pnl_df['basmom'] = pnl_df[["basmom60_ema", "basmom60_ema_xdemean", "basmom120_ema", "basmom120_ema_xdemean"]].sum(axis=1)
pnl_df = pnl_df[['carry', 'carrymom', 'basmom']]

In [ ]:
sns.heatmap(pnl_w_df[corr_cutoff:].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch 
from scipy.spatial.distance import pdist
import pylab

corr = pnl_w_df[corr_cutoff:].corr()

Z = sch.linkage(corr, 'complete')
print(Z[0])

# c, coph_dists = sch.cophenet(Z, pdist(corr))
# print(c)
plt.figure(figsize=(25, 10))
labelsize=20
ticksize=15
plt.title('Hierarchical Clustering Dendrogram for China Futures', fontsize=labelsize)
plt.xlabel('products', fontsize=labelsize)
plt.ylabel('distance', fontsize=labelsize)
sch.dendrogram(
    Z,
    leaf_rotation=90.,  # rotates the x axis labels
    leaf_font_size=8.,  # font size for the x axis labels
    labels = corr.columns
)
pylab.yticks(fontsize=ticksize)
pylab.xticks(rotation=-90, fontsize=ticksize)
plt.savefig('dendogram_'+'china_fut'+'.png')
plt.show()

In [ ]:
iplot(pnl_df[['combo']].cumsum())

In [ ]:
import pypfopt
from pypfopt import plotting
from pypfopt import risk_models
from pypfopt import EfficientFrontier
from pypfopt import expected_returns
#   
dpnl = pnl_df[[col for col in pnl_df.columns]]['2018-01-01':]

if 'combo' in dpnl.columns:
    dpnl = dpnl.drop(columns=['combo'])
dpnl = dpnl.fillna(0)

strat_std = dpnl.std()

dpnl = dpnl/dpnl.std()

#risk_models.sample_cov(dpnl, returns_data=True, frequency=244)

max_sharpe = False

if max_sharpe:
    mu = expected_returns.mean_historical_return(dpnl, returns_data=True, frequency=244, compounding=False)
else:
    mu = None

S = risk_models.CovarianceShrinkage(dpnl, returns_data=True, frequency=244).ledoit_wolf()
ef = EfficientFrontier(mu, S, weight_bounds= (0, 1)) 

fact_idx = {}
for fact in dpnl.columns:
    fact_idx[fact] = ef.tickers.index(fact)

#ef.add_constraint(lambda w: w[fact_idx['ryield_ema_ts']] + w[fact_idx['ryield_ema_xdemean']] >= 0.3)
#ef.add_constraint(lambda w: w[fact_idx['ryield_ema_ts']] + w[fact_idx['ryield_ema_xdemean']] <= 0.4)
#ef.add_constraint(lambda w: w[fact_idx['ryield_ema_ts']] + w[fact_idx['ryield_ema_xdemean']] >= 0.3)
# ef.add_constraint(lambda w: w[fact_idx['tsmom']] + w[fact_idx['macro2']] <= 0.24)

#ef = EfficientFrontier(mu, S, weight_bounds=(0,1))
if max_sharpe:
    port_weights = ef.max_sharpe()
else:
    port_weights = ef.min_volatility()

#port_weights = ef.clean_weights()
port_weights = pd.Series(port_weights)
print("port_weights=\n%s\n" % port_weights)
final_weights = port_weights.div(strat_std)
print("final_weights=\n%s\n" % final_weights)
# mu = expected_returns.mean_historical_return(df)
# S = risk_models.sample_cov(df)

# # Optimize for maximal Sharpe ratio
# ef = EfficientFrontier(mu, S)
# weights = ef.max_sharpe()
# ef.portfolio_performance(verbose=True)

In [ ]:
dpnl = pnl_df[[col for col in pnl_df.columns if 'mom_' in col]]['2015-01-01':] #.drop(columns=['combo'])
dpnl = dpnl/dpnl.std()
#risk_models.sample_cov(dpnl, returns_data=True, frequency=244)

mu = None
#mu = expected_returns.ema_historical_returns(dpnl, returns_data=True, frequency=244, span=500)
S = risk_models.CovarianceShrinkage(dpnl, returns_data=True, frequency=244).ledoit_wolf()
ef = EfficientFrontier(mu, S, weight_bounds=(0, 1)) 

fact_idx = {}

for fact in dpnl.columns:
    fact_idx[fact] = ef.tickers.index(fact)

#ef.add_constraint(lambda w: w[fact_idx['ryield_ema_ts']] + w[fact_idx['ryield_ema_xdemean']] >= 0.3)
#ef.add_constraint(lambda w: w[fact_idx['ryield_ema_ts']] + w[fact_idx['ryield_ema_xdemean']] <= 0.4)
#ef.add_constraint(lambda w: w[fact_idx['ryield_ema_ts']] + w[fact_idx['ryield_ema_xdemean']] >= 0.3)
# ef.add_constraint(lambda w: w[fact_idx['tsmom']] + w[fact_idx['macro2']] <= 0.24)

#ef = EfficientFrontier(mu, S, weight_bounds=(0,1))
ef.min_volatility()
#ef.max_sharpe()
port_weights = ef.clean_weights()
port_weights

In [ ]:
lead_lag_config = {
    'll_left': -20,
    'll_right': 120,
    'll_spacing': 5,
    'll_sub_win': [(datetime.date(2008, 1, 1), datetime.date(2018, 12, 31)), 
                   (datetime.date(2019, 1, 1), datetime.date(2024, 12, 31)),],
}

ll_keys = ['fullsample'] + ['%s:%s' % (sd.strftime('%Y-%b-%d'), ed.strftime('%Y-%b-%d')) for sd, ed in lead_lag_config['ll_sub_win']]


ll_left = lead_lag_config['ll_left']
ll_right = lead_lag_config['ll_right']
spacing = lead_lag_config['ll_spacing']

leadlag_df = bt_metrics.lead_lag(ll_limit_left=ll_left, 
                                 ll_limit_right=ll_right,
                                 ll_sub_windows=lead_lag_config['ll_sub_win'])

fig, ax = plt.subplots(len(ll_keys), 1)
fig.set_figheight(15)
fig.set_figwidth(10)

for i, key in enumerate(ll_keys):
    ts = leadlag_df['leadlag_sharpes'].loc[key]
    ts.plot(kind='bar', ax = ax[i], title = f'lead_lag: {key}')
    new_ticks = np.linspace(ll_left, ll_right, (ll_right-ll_left)//spacing+1)
    ax[i].set_xticks(np.interp(new_ticks, ts.index, np.arange(ts.size)))
    ax[i].set_xticklabels(new_ticks)
    ax[i].axvline(x=-ll_left, color='red', linestyle='--')
plt.show()

fig = plt.figure()
ax = fig.add_subplot(111)
ls_pnl = bt_metrics.long_short_pnl()
for key in ls_pnl:
    ax.plot(ls_pnl[key]['portfolio_cumpnl'].index, ls_pnl[key]['portfolio_cumpnl'].values, '-', label=key)
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, bbox_to_anchor=(1.04, 1), loc='upper left')
ax.grid()
plt.title("long-short pnl")
plt.show()

lagged = bt_metrics.lagged_pnl(lags=[1, 5, 10, 20, 30, 60, 75, 80])
lagged['cumpnl'].plot()
#print('lagged PNL\n', lagged['sharpe'])
plt.grid()
plt.title('lagged pnl')
plt.show()

smoothed = bt_metrics.smoothed_pnl(smooth_hls=[1, 5, 10, 20, 30, 60, 75, 80])
smoothed['cumpnl'].plot(figsize=(8, 6))
#print('smoothed PNL\n', smoothed['sharpe'])
plt.grid()
plt.title('smoothed pnl')
plt.show()

#tilt_timing = bt_metrics.tilt_timing(tilt_rolling_window=1*244) # default 3 years  tilt_rolling_window = 3 * 244 

seasonal_pnl = bt_metrics.seasonal_pnl()
cumpnl = seasonal_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('seasonal sharpe stats\n', seasonal_pnl['sharpe_stats'])
plt.grid()
plt.title('monthly pnl')
plt.show()


monthday_pnl = bt_metrics.monthday_pnl()
cumpnl = monthday_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('monthday sharpe stats\n', monthday_pnl['sharpe_stats'])
plt.grid()
plt.title('monthday pnl')
plt.show()


week_pnl = bt_metrics.week_pnl()
cumpnl = week_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('week sharpe stats\n', week_pnl['sharpe_stats'])
plt.grid()
plt.title('weekday pnl')
plt.show()


annual_pnl = bt_metrics.annual_pnl()
cumpnl = annual_pnl['cumlog_pnl']
cumpnl.set_index(cumpnl.index.astype('str')).plot(rot=30, figsize = (8, 6))
#print('annual sharpe stats\n', annual_pnl['sharpe_stats'])
plt.grid()
plt.title('annual pnl')
plt.show()

annual_pnl['cumlog_pnl'].mean(axis=1).plot()
plt.grid()
plt.title('annual averaged profile')
plt.show()

# turnover = bt_metrics.turnover()
# print(turnover)

# batch feature exploration

In [ ]:
feature_list = [
#     'margin_hrc_sh', 
    'strip_hsec',
    'strip_3.0x685',
    'pipe_1.5x3.25',    
    'hrc_sh',
    'crc_sh',
    'billet_ts',
    'macf_cfd',
#     'gi_0.5_sh',
#     'hsec_400x200',
#     'highwire_6.5',
#     'angle_50x5',
#     'ibeam_25',
#     'channel_16',

#     'import_arb', 'pbf_prem', 'plt65_62',
#     'io_laytime_45ports', 'io_inv_imp_31ports',
#     'io_invdays_imp_mill(64)', 'io_inv_mill(64)', 'io_inv_imp_mill',
#     'io_removal_port_41',
#     'io_loading_14ports_ausbzl',
]

udf = spot_df[feature_list].dropna(how='all')
# lunar_seasonal = True

# if lunar_seasonal:
#     seasonal_signal = tstool.lunar_label(udf)
#     seasonal_signal = tstool.seasonal_group_score(
#         seasonal_signal, score_cols=feature_list, yr_col='lunar_cny',
#         group_col='lunar_wks', min_obs=3, backward=2, forward=2, rolling_years=3)
#     seasonal_signal = seasonal_signal.reindex(index=df.index).ffill()

for feature in udf.columns:
    dataseries.plot_seasonal_df(udf[feature].dropna(), cutoff='2018-01-01', title=feature)
    
signal_raw = udf[feature_list].reindex(index=df.index).ffill()


In [ ]:
cutoff = '2012-01-01'
signal_func = 'qtl'
param_rng = [20, 42, 2]
signal_cap = None # [-2, 2]
product_list = ['rb', 'hc', 'i', 'j', 'jm', 'SF', 'FG', ] # 'v', 'cu', 'al', 'ss', 'UR', 'SA', 'ru'

for asset in product_list:
    if '_' in asset:
        price_ts = (1 + beta_ret_dict[asset]).cumprod().to_frame('price')[cutoff:]
    else:
        price_ts = df[(asset, 'c1', 'close')].dropna().to_frame('price')[cutoff:]
    pnl_list = [price_ts]
    for feature in feature_list:
        feature_ts = udf[feature].reindex(index=price_ts.index).ffill()
        #feature_ts = feature_ts.pct_change(5)
        #feature_ts = tstool.lunar_yoy(feature_ts, group_col='lunar_days', func='pct_change')
        #feature_ts = tstool.seasonal_score(feature_ts.to_frame())
        signal_ts = calc_conv_signal(feature_ts, signal_func=signal_func, param_rng=param_rng, signal_cap=signal_cap)
        asset_df = pd.concat([price_ts, signal_ts], axis=1)
        asset_df.columns = ['price', 'signal']
        asset_df['signal'] = asset_df['signal'].apply(lambda x: x).ffill()
        asset_df = asset_df.dropna(subset=['price'])
        asset_df['position'] = (asset_df['signal']/asset_df['price'].pct_change().rolling(20).std()).shift(1).fillna(0)
        asset_df['pnl'] = (asset_df['position'].shift(1) * asset_df['price'].pct_change()).fillna(0)
        
        sr = np.sqrt(244) * asset_df['pnl'].mean()/asset_df['pnl'].std()
        pnl_per_trade = 100 * 100 * asset_df['pnl'].mean()/asset_df['position'].diff().abs().mean()
        turnover = 100 * asset_df['position'].diff().abs().mean()/asset_df['position'].abs().mean()
        print(f'{asset}:{feature} -> SR: {sr:.2f} -- PNL per trade: {pnl_per_trade:.2f} -- Turnover: {turnover:.2f}')
        pnl_list.append(asset_df['pnl'].cumsum().to_frame(feature))
    pnl_df = pd.concat(pnl_list, axis=1)
    dataseries.plot_df_on_2ax(pnl_df, left_on=feature_list, right_on=['price'])
    

# signal grid search run

In [ ]:
signal_list = []

for feature in feature_list:
    for win in [20, 40, 60, 80, 120, 240]:
        signal_name = f"{feature}:ma:{win}"
        signal_raw[signal_name] = signal_raw[feature] - signal_raw[feature].rolling(win).mean()
        signal_raw[signal_name] = dh.risk_normalized(signal_raw[signal_name], 60)
        signal_list.append(signal_name)
        
        signal_name = f"{feature}:ewmac:{win}"
        signal_raw[signal_name] = dh.ewmac(signal_raw[feature], win_s=win/10, ls_ratio=2)
        signal_raw[signal_name] = dh.risk_normalized(signal_raw[signal_name], 60)
        signal_list.append(signal_name)

#         signal_name = f"{feature}:convewm:{win}"
#         signal_raw[signal_name] = dh.conv_ewm(signal_raw[feature], h1s=[win//10, win//10*2], h2s=[win//10*3, win//10*6])
#         signal_raw[signal_name] = dh.risk_normalized(signal_raw[signal_name], 60)
#         signal_list.append(signal_name)
        
        signal_name = f"{feature}:zscore:{win}"
        signal_raw[signal_name] = dh.zscore_roll(signal_raw[feature], win=win)
        signal_list.append(signal_name) 
        
        signal_name = f"{feature}:zscore_dff20:{win}"
        signal_raw[signal_name] = dh.zscore_roll(signal_raw[feature].diff(20), win=win)
        signal_list.append(signal_name) 
        
        signal_name = f"{feature}:qtl:{win}"
        signal_raw[signal_name] = dh.pct_score(signal_raw[feature], win=win)*2
        signal_list.append(signal_name) 
        
        signal_name = f"{feature}:qtl_dff20:{win}"
        signal_raw[signal_name] = dh.pct_score(signal_raw[feature].diff(20), win=win)*2
        signal_list.append(signal_name)
        
        signal_name = f"{feature}:lunar_wks_score:{win}"
        signal_raw[signal_name] = seasonal_signal[feature]
        signal_list.append(signal_name)
        
        signal_prefix = f"{feature}:seasonal_score"
        signal_raw[signal_prefix] = tstool.seasonal_score(signal_raw[feature].to_frame())
        signal_name = f"{signal_prefix}_pct:{win}"
        signal_raw[signal_name] = dh.pct_score(signal_raw[feature], win=win)*2
        signal_list.append(signal_name) 
        signal_name = f"{signal_prefix}_zscore:{win}"
        signal_raw[signal_name] = dh.zscore_roll(signal_raw[feature], win=win)
        signal_list.append(signal_name)
        
        signal_prefix = f"{feature}:yoy"
        signal_raw[signal_prefix] = signal_raw[feature]/signal_raw[feature].shift(244)-1
        signal_name = f"{signal_prefix}_pct:{win}"
        signal_raw[signal_name] = dh.pct_score(signal_raw[feature], win=win)*2
        signal_list.append(signal_name) 
        signal_name = f"{signal_prefix}_zscore:{win}"
        signal_raw[signal_name] = dh.zscore_roll(signal_raw[feature], win=win)
        signal_list.append(signal_name)

        signal_prefix = f"{feature}:lunar_yoy"
        signal_raw[signal_prefix] = tstool.lunar_yoy(signal_raw[feature], group_col='lunar_days', func='pct_change')
        signal_name = f"{signal_prefix}_pct:{win}"
        signal_raw[signal_name] = dh.pct_score(signal_raw[feature], win=win)*2
        signal_list.append(signal_name) 
        signal_name = f"{signal_prefix}_zscore:{win}"
        signal_raw[signal_name] = dh.zscore_roll(signal_raw[feature], win=win)
        signal_list.append(signal_name)
        
signal_raw = signal_raw.reindex(index=df.index).ffill()

In [ ]:
product_list = ['rb', 'hc', 'i', 'j', 'jm', 'v', 'FG', 'SM', 'SF']
cutoff = pd.Timestamp('2012-07-01')

for sig in signal_list:
    print(sig)
    pnl_by_asset = {}
    pnl_df = pd.DataFrame()
    pos_df = pd.DataFrame()
    for asset in product_list:
        signal = signal_raw[sig]
        asset_df = pd.concat([df[(asset, 'c1', 'close')], signal], axis=1)
        asset_df.columns = ['price', 'signal']
        asset_df['signal'] = asset_df['signal'].apply(lambda x: x).ffill()
        asset_df = asset_df.dropna(subset=['price']).ffill()
        asset_df['position'] = (asset_df['signal']/asset_df['price'].pct_change().rolling(20).std()).shift(1).fillna(0)
        asset_df['pnl'] = (asset_df['position'].shift(1) * asset_df['price'].pct_change()).fillna(0)
        
        sr = np.sqrt(244) * asset_df['pnl'].mean()/asset_df['pnl'].std()
        pnl_per_trade = 100 * 100 * asset_df['pnl'].mean()/asset_df['position'].diff().abs().mean()
        turnover = 100 * asset_df['position'].diff().abs().mean()/asset_df['position'].abs().mean()
        print(f'{asset} -> SR: {sr:.2f} -- PNL per trade: {pnl_per_trade:.2f} -- Turnover: {turnover:.2f}')
        
        pnl_by_asset[asset] = asset_df
        pnl_df[asset] = asset_df['pnl']
        pos_df[asset] = asset_df['position']
    pnl_df = pnl_df.fillna(0)
    pos_df = pos_df.ffill()
    total_sr = = np.sqrt(244) * pnl_df.sum(axis=1).mean()/pnl_df.sum(axis=1).std()
    print(f'Total SR: {total_sr:.2f}')
    
    cumpn; = pnl_df.cumsum()
    cumpnl.plot()
    plt.title(sig)
    plt.show()
    cum_pnl.sum(axis=1).plot()
    plt.title(sig)
    plt.show()
    

# Signal portfolio

In [ ]:
signal_dict_full = {
    'i': [
        ('io_removal_lvl_fast', 1.0), 
        #('io_removal_lyoy_mom', 1.0),
        
        ('io_inv_mill(64)_lvl_fast', 0.5),
        #('io_inv_mill(64)_lyoy_mom', 0.5),
        
        ('io_invdays_imp_mill(64)_lvl_fast', 0.5),
        #('io_invdays_imp_mill(64)_lyoy_mom', 0.5),
        
#         ('steel_social_inv_lvl_fast', 1.0/1.0),
#         ('rebar_inv_social_lyoy_fast', 0.25/1.0),
#         ('wirerod_inv_social_lyoy_fast', 0.25/1.0),
#         ('hrc_inv_social_lyoy_fast', 0.25/1.0),
#         ('crc_inv_social_lyoy_fast', 0.25/1.0),
        
        ('margin_lvl_fast', 1.0),
        ('strip_hsec_lvl_mid', 1.0),
        ('macf_cfd_lvl_mid', 1.0),
#         ('pbf_prem_yoy', 0.5/15),
#         ('cons_steel_lyoy_slow', 1.0/1.5),
#         ('sea_export_arb_lvl_mid', 1.0/1.4),
    ],
    'rb': [
        ('io_removal_lvl_fast', 1.0), 
        #('io_removal_lyoy_mom', 1.0),
        
        ('io_inv_mill(64)_lvl_fast', 1.0),
        #('io_inv_mill(64)_lyoy_mom', 1.0),
        
#         ('rebar_inv_social_lyoy_fast', 1.0),
#         ('wirerod_inv_social_lyoy_fast', 1.0),
        
        ('margin_lvl_fast', 1.0),
        ('strip_hsec_lvl_mid', 1.0),    
    ],
    'hc': [
        ('io_removal_lvl_fast', 1.0), 
        #('io_removal_lyoy_mom', 1.0),
        
        ('io_inv_mill(64)_lvl_fast', 1.0),
        #('io_inv_mill(64)_lyoy_mom', 1.0),
        
#         ('hrc_inv_social_lyoy_fast', 1.0),
#         ('crc_inv_social_lyoy_fast', 1.0),        
        ('margin_lvl_fast', 1.0),
        ('strip_hsec_lvl_mid', 1.0),  
    ],
    'j': [
        #('io_removal_lvl_fast', 1.0), 
        #('io_removal_lyoy_mom', 1.0),
        
        #('io_inv_mill(64)_lvl_fast', 1.0),
        #('io_inv_mill(64)_lyoy_mom', 1.0),
        
#         ('steel_social_inv_lvl_fast', 1.0),
        
        ('margin_lvl_fast', 1.0),
        ('strip_hsec_lvl_mid', 1.0),    
    ],
    'jm': [
        #('io_inv_mill(64)_lvl_fast', 1.0),
        #('io_inv_mill(64)_lyoy_mom', 1.0),
        
#         ('steel_social_inv_lvl_fast', 1.0),
        
        ('margin_lvl_fast', 1.0),
        ('strip_hsec_lvl_mid', 1.0),     
    ],
    'FG': [
        #('io_inv_mill(64)_lvl_fast', 1.0),
        #('io_inv_mill(64)_lyoy_mom', 1.0),
        
        ('margin_lvl_fast', 1.0),
        ('strip_hsec_lvl_mid', 1.0),    
    ],
}

In [ ]:
signal_dict = signal_dict_full

signal_diagnosis = False

pnl_dict = {}
pos_dict = {}

for asset in ['i', 'rb', 'hc', 'j', 'jm']:
    if '_' in asset:
        price_ts = (1 + beta_ret_dict[asset]).cumprod().to_frame('price')
    else:
        price_ts = df[(asset, 'c1', 'close')].dropna().to_frame('price')
    pnl_list = []
    pos_list = []
    for idx, (feature_name, weight) in enumerate(signal_dict[asset]):
        feature, signal_func, param_rng, proc_func, chg_func, bullish, freq = signal_repo[feature_name]
        if freq == 'price':
            feature_ts = spot_df[feature].ffill().reindex(index=price_ts.index).ffill()
        elif len(freq) > 0:
            feature_ts = spot_df[feature].ffill().reindex(index=pd.date_range(start=df.index[0], end=df.index[-1], freq=freq)).ffill()
        else:
            feature_ts = spot_df[feature].dropna()
        
        if 'yoy' in proc_func:
            if 'lunar' in proc_func:
                label_func = lunar_label
                label_args = {}
            else:
                label_func = calendar_label
                label_args = {'anchor_date': {'month': 1, 'day': 1}}
            if '_wk' in proc_func:
                group_col = 'label_wk'
            else:
                group_col = 'label_day'
            feature_ts = yoy_generic(feature_ts, label_func=label_func, group_col='label_day', func=chg_func, label_args=label_args)
        elif 'df' in proc_func:
            n_diff = int(proc_func[2:])
            feature_ts = getattr(feature_ts, chg_func)(n_diff)
    
        if signal_func == 'seasonal_score_w':
            signal_ts = seasonal_score(feature_ts.to_frame(), backward=10, forward=10, rolling_years=3, min_obs=10).reindex(index=df.index).ffill()
        elif signal_func == 'seasonal_score_d':
            signal_ts = seasonal_score(feature_ts.to_frame(), backward=15, forward=15, rolling_years=3, min_obs=30)
        elif len(signal_func)>0:
            feature_ts = feature_ts.reindex(index=df.index).ffill()
            signal_ts = calc_conv_signal(feature_ts, signal_func=signal_func, param_rng=param_rng, signal_cap=signal_cap)
        else:
            signal_ts = feature_ts.reindex(index=df.index).ffill()
            
        if not bullish:
            signal_ts = -signal_ts
            
        asset_df = pd.concat([price_ts, signal_ts], axis=1)
        asset_df.columns = ['price', 'signal']
        asset_df['signal'] = asset_df['signal'].apply(lambda x: x).ffill()
        asset_df = asset_df.dropna(subset=['price'])
        asset_df['position'] = (weight*asset_df['signal']/asset_df['price'].pct_change().rolling(20).std()).shift(1).fillna(0)
        asset_df['pnl'] = (asset_df['position'].shift(1) * asset_df['price'].pct_change()).fillna(0)
        
        std = asset_df['pnl'].std()
        sr = np.sqrt(244) * asset_df['pnl'].mean()/asset_df['pnl'].std()
        pnl_per_trade = 100 * 100 * asset_df['pnl'].mean()/asset_df['position'].diff().abs().mean()
        turnover = 100 * asset_df['position'].diff().abs().mean()/asset_df['position'].abs().mean()
        print(f'{asset}:{feature_name} -> SR: {sr:.2f} -- PNL per trade: {pnl_per_trade:.2f} -- Turnover: {turnover:.2f}')
        pnl_list.append(asset_df['pnl'].to_frame(feature_name))
        pos_list.append(asset_df['position'].to_frame(feature_name))
        
    pnl_df = pd.concat(pnl_list, axis=1)
    pos_df = pd.concat(pos_list, axis=1)
    sum_pnl = pnl_df.sum(axis=1)
    sum_pos = pos_df.sum(axis=1)
    sr = np.sqrt(244) * sum_pnl.mean()/sum_pnl.std()
    pnl_per_trade = 100 * 100 * sum_pnl.mean()/sum_pos.diff().abs().mean()
    turnover = 100 * sum_pos.diff().abs().mean()/sum_pos.abs().mean()
    print(f'{asset}:total -> SR: {sr:.2f} -- PNL per trade: {pnl_per_trade:.2f} -- Turnover: {turnover:.2f}')
    
    print(pnl_df.std())
    pnl_dict[asset] = pnl_df
    pos_dict[asset] = pos_df
    pnl_df.cumsum().plot()
    plt.show()
    sum_pnl.cumsum().plot()
    plt.show()


In [ ]:
fill_backward = False
smooth_win = 1
sig_smooth = tstool.exp_smooth(df_in, hl = smooth_win, fill_backward=fill_backward)

demean = False
mean_win = 244
vol_win = 244
if demean:
    sig_scored = tstool.ts_score(sig_smooth, hl_mean=mean_win, min_obs_mean=mean_win, fill_backward_mean=fill_backward, 
                          hl_vol=vol_win, min_obs_vol=vol_win, fill_backward_vol=fill_backward)
else:
    sig_scored = tstool.ts_scale(sig_smooth, hl = vol_win, min_obs=vol_win, fill_backward=fill_backward)

#sig_scored = tstool.xs_score(sig_smooth, demean=demean, hl=vol_win)

signal_cap = 2.0

score_capped = tstool.cap(sig_scored, -signal_cap, signal_cap)
score_filled = tstool.filldown(score_capped, 2)
score = tstool.lag(score_filled, 1)


In [ ]:
vol_scale = 20
asset_vol = tstool.exp_smooth(df_pxchg**2, hl=vol_scale, fill_backward=fill_backward)**0.5
holding = score/asset_vol

commod_list = holding.columns #['hc']
btmetrics = MetricsBase(holdings = holding[commod_list], returns = df_pxchg[commod_list])